In [1]:
from langchain_community.document_loaders import PyPDFLoader

from sentence_transformers import SentenceTransformer

import chromadb

import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

C:\Users\Vivian\AppData\Local\Temp\ipykernel_52020\2314605415.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
c:\Users\Vivian\OneDrive\Documents\RAG Krish Naik\learn_rag\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

print(
    pytesseract.get_tesseract_version()
)

5.5.0.20241111


In [3]:
import torch

print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.11.0+cu128
12.8
True
NVIDIA GeForce RTX 4050 Laptop GPU


In [4]:
#Load Embedding Model

#strong semantic model.
model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4905.06it/s]


In [5]:
#Create ChromaDB
client = chromadb.PersistentClient(
    path="./resume_db"
)

collection = client.get_or_create_collection(
    name="resume_collection",
    metadata={
        "hnsw:space": "cosine"
    }
)

In [6]:
#Load  PDFs
import os

pdf_files = [
    os.path.join("resumes", f)
    for f in os.listdir("resumes")
    if f.endswith(".pdf")
]

pdf_files

['resumes\\1603.00968.pdf',
 'resumes\\1603.04513.pdf',
 'resumes\\1603.07044.pdf',
 'resumes\\1604.00400.pdf',
 'resumes\\1605.07333.pdf',
 'resumes\\1605.07683.pdf',
 'resumes\\1605.08675.pdf',
 'resumes\\1606.00189.pdf',
 'resumes\\1606.04631.pdf',
 'resumes\\1606.05320.pdf',
 'resumes\\1607.06025.pdf',
 'resumes\\1608.06757.pdf',
 'resumes\\1609.00559.pdf',
 'resumes\\1610.00879.pdf',
 'resumes\\1610.07809.pdf',
 'resumes\\1611.00514.pdf',
 'resumes\\1611.02550.pdf',
 'resumes\\1611.03382.pdf',
 'resumes\\1611.04642.pdf',
 'resumes\\1611.04798.pdf',
 'resumes\\1612.05270.pdf',
 'resumes\\1612.08205.pdf',
 'resumes\\1701.00185.pdf',
 'resumes\\1701.02877.pdf',
 'resumes\\1701.03214.pdf',
 'resumes\\1701.05574.pdf',
 'resumes\\1701.06538.pdf',
 'resumes\\1701.09123.pdf',
 'resumes\\1702.03342.pdf',
 'resumes\\1703.02507.pdf',
 'resumes\\1703.06492.pdf',
 'resumes\\1703.07090.pdf',
 'resumes\\1704.00939.pdf',
 'resumes\\1704.05572.pdf',
 'resumes\\1704.05907.pdf',
 'resumes\\1704.0619

In [7]:
#Extract Text - Document structure
import fitz


from PIL import Image


def extract_content(pdf_path):

    doc = fitz.open(pdf_path)
    full_text = ""

    for page in doc:

        # ---------------------------
        # 1. Try direct text extraction
        # ---------------------------
        text = page.get_text("text").strip()

        # If enough text exists, use it
        if len(text) > 50:
            full_text += text + "\n"
            continue

        # ---------------------------
        # 2. Fallback to OCR
        # ---------------------------
        pix = page.get_pixmap(
            matrix=fitz.Matrix(3, 3),
            alpha=False
        )

        img = Image.frombytes(
            "RGB",
            [pix.width, pix.height],
            pix.samples
        )

        ocr_text = pytesseract.image_to_string(img)

        full_text += ocr_text + "\n"

    doc.close()

    return full_text

In [8]:
#Create embeddings
for pdf in pdf_files:

    content = extract_content(pdf)


    embedding = model.encode(
        content
    ).tolist()

    collection.add(
    ids=[pdf],          # <-- filename/path stored here
    documents=[content],
    embeddings=[embedding]
)

print("Knowledge Base Created")

Knowledge Base Created


In [9]:
#Similarity Search Function
def search_resumes(query, top_k=5):

    query_embedding = model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=top_k,
        include=["distances", "documents"]
    )

    return results

In [10]:
hr_input = "vivian"

In [11]:
#hr_input = "I need a web developer"

In [12]:


results = search_resumes(
    hr_input,
    top_k=5
)

results

{'ids': [['resumes\\1707.03764.pdf',
   'resumes\\1703.06492.pdf',
   'resumes\\1701.06538.pdf',
   'resumes\\1704.05572.pdf',
   'resumes\\1705.08142.pdf']],
 'embeddings': None,
 'documents': [['N-GrAM: New Groningen Author-proﬁling Model\nNotebook for PAN at CLEF 2017\nAngelo Basile, Gareth Dwyer, Maria Medvedeva,\nJosine Rawee, Hessel Haagsma, and Malvina Nissim\nUniversity of Groningen, Groningen, The Netherlands\n{a.basile,g.t.b.dwyer,j.n.rawee}@student.rug.nl,\nmedvmr@gmail.com,\n{hessel.haagsma,m.nissim}@rug.nl\nAbstract We describe our participation in the PAN 2017 shared task on Author\nProﬁling, identifying authors’ gender and language variety for English, Span-\nish, Arabic and Portuguese. We describe both the ﬁnal, submitted system, and\na series of negative results. Our aim was to create a single model for both gen-\nder and language, and for all language varieties. Our best-performing system (on\ncross-validated results) is a linear support vector machine (SVM) with word

In [13]:
# show results
for i, doc in enumerate(results["documents"][0]):

    print("=" * 80)
    print(f"Resume {i+1}")

    # Filename
    print(f"File Name      : {results['ids'][0][i]}")

    # Similarity Score
    distance = results["distances"][0][i]
    similarity = 1 - distance   # for cosine distance

    print(f"Distance Score : {distance:.4f}")
    print(f"Similarity     : {similarity:.4f}")

    print("=" * 80)

    print(doc[:1000])
    print()

Resume 1
File Name      : resumes\1707.03764.pdf
Distance Score : 0.8539
Similarity     : 0.1461
N-GrAM: New Groningen Author-proﬁling Model
Notebook for PAN at CLEF 2017
Angelo Basile, Gareth Dwyer, Maria Medvedeva,
Josine Rawee, Hessel Haagsma, and Malvina Nissim
University of Groningen, Groningen, The Netherlands
{a.basile,g.t.b.dwyer,j.n.rawee}@student.rug.nl,
medvmr@gmail.com,
{hessel.haagsma,m.nissim}@rug.nl
Abstract We describe our participation in the PAN 2017 shared task on Author
Proﬁling, identifying authors’ gender and language variety for English, Span-
ish, Arabic and Portuguese. We describe both the ﬁnal, submitted system, and
a series of negative results. Our aim was to create a single model for both gen-
der and language, and for all language varieties. Our best-performing system (on
cross-validated results) is a linear support vector machine (SVM) with word uni-
grams and character 3- to 5-grams as features. A set of additional features, includ-
ing POS tags, addition

In [14]:
#Test for pdf with image
'''
content = extract_content("resumes/Shambhavi_Resume.pdf")

print(len(content))
print(content[:1000])
'''

'\ncontent = extract_content("resumes/Shambhavi_Resume.pdf")\n\nprint(len(content))\nprint(content[:1000])\n'